In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML
import base64

PROJECT_ROOT = Path(".").resolve()
MAKE_MODEL_DIR    = PROJECT_ROOT / "traffic_data" / "make_model"
LICENSE_PLATE_DIR = PROJECT_ROOT / "traffic_data" / "license_plates"
TSV_PATH = MAKE_MODEL_DIR / "make_model.tsv"

N_ROWS        = 100   # number of rows to review
SHOW_REVIEWED = False  # False → unreviewed only | True → reviewed only
SHOW_FAILED   = False  # False → successful lookups | True → failed lookups (for OCR correction)

df = pd.read_csv(TSV_PATH, sep="\t", dtype=str).fillna("")

if SHOW_FAILED:
    df = df[df["api_success"] == "0"]
else:
    df = df[df["api_success"] == "1"]

if "reviewed" not in df.columns:
    df["reviewed"] = ""

if SHOW_REVIEWED:
    df = df[df["reviewed"] == "1"]
else:
    df = df[df["reviewed"] != "1"]

df = df.sort_values(["date", "timestamp"], ascending=False).head(N_ROWS).reset_index(drop=True)
mode  = "failed" if SHOW_FAILED else "successful"
label = "reviewed" if SHOW_REVIEWED else "unreviewed"
print(f"{len(df)} {label} {mode} events (most recent)")

4 unreviewed successful events (most recent)


In [2]:
import ipywidgets as widgets

display(HTML("""<style>
.review-row {
    background: #111 !important;
    border: 1px solid #333 !important;
    border-radius: 6px;
    margin-bottom: 8px !important;
    padding: 4px !important;
}
.review-row .widget-vbox,
.review-row .widget-hbox,
.review-row .widget-label,
.review-row .widget-checkbox {
    background: transparent !important;
}
.review-row input[type="checkbox"] {
    width: 22px;
    height: 22px;
    cursor: pointer;
}
</style>"""))

IMG_WIDTH  = 960
GAP        = 8
PLATE_W    = 300  # width of superimposed plate crop

def img_tag(path: Path, width: int = IMG_WIDTH) -> str:
    if not path.exists():
        return f"<div style='width:{width}px;height:{width//3}px;background:#222;display:flex;align-items:center;justify-content:center;color:#555'>no image</div>"
    data = base64.b64encode(path.read_bytes()).decode()
    return f'<img src="data:image/jpeg;base64,{data}" style="width:{width}px;border-radius:4px;display:block">'

def find_plate_crop(date: str, timestamp: str, plate: str) -> Path | None:
    date_compact = date.replace("_", "")
    folder = LICENSE_PLATE_DIR / date
    if not folder.exists():
        return None
    matches = list(folder.glob(f"{date_compact}_{timestamp}_*_{plate}_*.jpg"))
    return matches[0] if matches else None

def render_card(row) -> str:
    date_dir = row["date"]
    ef = img_tag(MAKE_MODEL_DIR / date_dir / row["filename_ef"]) if row["filename_ef"] else img_tag(Path("__missing__"))
    wf = img_tag(MAKE_MODEL_DIR / date_dir / row["filename_wf"]) if row["filename_wf"] else img_tag(Path("__missing__"))

    plate_crop_path = find_plate_crop(date_dir, row["timestamp"], row["plate"])
    if plate_crop_path:
        plate_data = base64.b64encode(plate_crop_path.read_bytes()).decode()
        plate_overlay = f"""
        <img src="data:image/jpeg;base64,{plate_data}"
             style="position:absolute;
                    width:{PLATE_W}px;
                    left:550px;
                    bottom:10px;
                    transform:translateX(-50%);
                    border-radius:4px;
                    border:2px solid #fff;
                    box-shadow:0 4px 16px rgba(0,0,0,0.8)">
        """
    else:
        plate_overlay = ""

    if row["api_success"] == "0":
        err = row.get("api_error", "") or "unknown error"
        meta = f"<span style='color:#e55'>lookup failed</span> &nbsp;·&nbsp; <span style='color:#777;font-size:12px'>{err}</span>"
    else:
        meta = f"<b>{row['year']} {row['make']} {row['model']}</b> &nbsp;·&nbsp; <span style='color:#aaa'>{row['trim']}</span>"

    return f"""
    <div style='padding:12px;font-family:sans-serif;color:#eee'>
      <div style='margin-bottom:8px;font-size:14px'>
        <span style='font-size:16px;font-weight:bold;letter-spacing:1px'>{row['plate']}</span>
        &nbsp;&nbsp;{meta}
        <span style='float:right;color:#555;font-size:12px'>{date_dir} &nbsp; {row['timestamp']}</span>
      </div>
      <div style='position:relative;display:inline-flex;gap:{GAP}px'>
        <div>{ef}<div style='font-size:11px;color:#555;margin-top:2px;text-align:center'>EF</div></div>
        <div>{wf}<div style='font-size:11px;color:#555;margin-top:2px;text-align:center'>WF</div></div>
        {plate_overlay}
      </div>
    </div>
    """

checkboxes = []  # (date, timestamp, plate, cb_tsv, cb_cache, tb_plate, filename_ef, filename_wf)

rows_out = []
for _, row in df.iterrows():
    cb_tsv = widgets.Checkbox(value=False, description='', indent=False,
                              layout=widgets.Layout(width='30px'))
    cb_cache = widgets.Checkbox(value=False, description='', indent=False,
                                layout=widgets.Layout(width='30px'))
    tb_plate = widgets.Text(value='', placeholder='',
                            layout=widgets.Layout(width='70px'))
    cb_col = widgets.VBox(
        [
            cb_tsv,
            widgets.HTML('<div style="color:#888;font-size:12px;text-align:center;margin-top:4px">TSV</div>'),
            widgets.HTML('<div style="margin-top:10px"></div>'),
            cb_cache,
            widgets.HTML('<div style="color:#888;font-size:12px;text-align:center;margin-top:4px">Cache</div>'),
            widgets.HTML('<div style="margin-top:10px"></div>'),
            tb_plate,
            widgets.HTML('<div style="color:#888;font-size:10px;text-align:center;margin-top:4px">Plate</div>'),
        ],
        layout=widgets.Layout(align_items='center', min_width='78px', width='78px', margin='12px 8px 0 12px'),
    )
    checkboxes.append((row["date"], row["timestamp"], row["plate"], cb_tsv, cb_cache, tb_plate,
                       row["filename_ef"], row["filename_wf"]))
    hbox = widgets.HBox(
        [cb_col, widgets.HTML(render_card(row))],
        layout=widgets.Layout(align_items='flex-start'),
    )
    hbox.add_class('review-row')
    rows_out.append(hbox)

display(widgets.VBox(rows_out))

In [ ]:
import json
import shutil
import subprocess

reviewed_keys = {(d, ts, p) for d, ts, p, cb_tsv, cb_cache, tb_plate, fnef, fnwf in checkboxes}
flagged_keys  = {(d, ts, p) for d, ts, p, cb_tsv, cb_cache, tb_plate, fnef, fnwf in checkboxes if cb_tsv.value}
ocr_keys      = {p           for d, ts, p, cb_tsv, cb_cache, tb_plate, fnef, fnwf in checkboxes if cb_cache.value}
plate_fixes   = {(d, ts, p): tb_plate.value.strip().upper()
                 for d, ts, p, cb_tsv, cb_cache, tb_plate, fnef, fnwf in checkboxes
                 if tb_plate.value.strip()}
filenames     = {(d, ts, p): (fnef, fnwf)
                 for d, ts, p, cb_tsv, cb_cache, tb_plate, fnef, fnwf in checkboxes}

# ── File copies for plate corrections ─────────────────────────────────────────
for key, new_plate in plate_fixes.items():
    d, ts, old_plate = key
    fnef, fnwf = filenames[key]

    for fn in filter(None, [fnef, fnwf]):
        src = MAKE_MODEL_DIR / d / fn
        dst = MAKE_MODEL_DIR / d / fn.replace(old_plate, new_plate)
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

    date_compact = d.replace("_", "")
    crop_dir = LICENSE_PLATE_DIR / d
    if crop_dir.exists():
        for src in crop_dir.glob(f"{date_compact}_{ts}_*_{old_plate}_*.jpg"):
            dst = crop_dir / src.name.replace(old_plate, new_plate)
            if not dst.exists():
                shutil.copy2(src, dst)

# ── TSV ───────────────────────────────────────────────────────────────────────
full_df = pd.read_csv(TSV_PATH, sep="\t", dtype=str).fillna("")

for col in ("reviewed", "flagged"):
    if col not in full_df.columns:
        full_df[col] = ""

# Auto-flag reviewed rows where lookup failed (no plate correction entered).
full_df["_key"] = list(zip(full_df["date"], full_df["timestamp"], full_df["plate"]))
auto_failed = set(
    full_df.loc[
        (full_df["api_success"] == "0") &
        full_df["_key"].isin(reviewed_keys) &
        ~full_df["_key"].isin(plate_fixes),
        "_key"
    ]
)
flagged_keys |= auto_failed
ocr_keys     |= {p for _, _, p in auto_failed}
full_df.drop(columns=["_key"], inplace=True)

def apply_flags(r):
    key = (r["date"], r["timestamp"], r["plate"])
    if key in reviewed_keys:
        r["reviewed"] = "1"
    if key in flagged_keys or key in plate_fixes:
        r["flagged"] = "1"
    return r

full_df = full_df.apply(apply_flags, axis=1)
full_df.to_csv(TSV_PATH, sep="\t", index=False)

# ── Cache ─────────────────────────────────────────────────────────────────────
CACHE_PATH = MAKE_MODEL_DIR / "vehicle_cache_platetovin.json"

if ocr_keys or plate_fixes:
    cache = json.loads(CACHE_PATH.read_text())
    for plate in ocr_keys:
        if plate in cache:
            cache[plate]["ocr_error"] = 1
    for (d, ts, old_plate) in plate_fixes:
        if old_plate in cache:
            cache[old_plate]["ocr_error"] = 1
    CACHE_PATH.write_text(json.dumps(cache, indent=2))

# ── Corrections queue ──────────────────────────────────────────────────────────
CORRECTIONS_PATH = MAKE_MODEL_DIR / "plate_corrections.json"
if plate_fixes:
    pending = set(json.loads(CORRECTIONS_PATH.read_text())) if CORRECTIONS_PATH.exists() else set()
    pending.update(plate_fixes.values())
    CORRECTIONS_PATH.write_text(json.dumps(sorted(pending), indent=2))

# ── This review summary ────────────────────────────────────────────────────────
manual_flagged = len(flagged_keys) - len(auto_failed)
print("[This Review]")
print(f"TSV: {len(reviewed_keys)} marked reviewed, {manual_flagged} flagged (manual), {len(auto_failed)} flagged (auto: failed lookup).")
if plate_fixes:
    print(f"Plate corrections ({len(plate_fixes)}):")
    for (d, ts, old), new in plate_fixes.items():
        print(f"  {d} {ts}  {old} → {new}  [queued]")
print(f"Cache: {len(ocr_keys)} plates marked ocr_error." if ocr_keys else "Cache: no OCR errors marked.")

# ── DB Totals ─────────────────────────────────────────────────────────────────
_df = pd.read_csv(TSV_PATH, sep="\t", dtype=str).fillna("")
if "thresholded" not in _df.columns:
    _df["thresholded"] = ""

total_reviewed  = (_df["reviewed"] == "1").sum()
total_validated = (
    (_df["reviewed"] == "1") &
    (_df["flagged"] != "1") &
    (_df["thresholded"] != "1")
).sum()
pct = 100 * total_validated / total_reviewed if total_reviewed else 0

print("\n[DB Totals]")
print(f"{total_reviewed} rows reviewed.")
print(f"{total_validated}/{total_reviewed} rows validated. ({pct:.1f}% validation rate)")

# ── Push to Mini ───────────────────────────────────────────────────────────────
MINI     = "jrill@192.168.50.100"
MINI_TSV = f"{MINI}:~/Documents/traffic_project/traffic_data/make_model/make_model.tsv"

result = subprocess.run(
    ["rsync", "-az", str(TSV_PATH), MINI_TSV],
    capture_output=True, text=True, timeout=30,
)
if result.returncode == 0:
    print("\nPushed make_model.tsv to Mini.")
else:
    print(f"\nMini push failed (SSH key auth required): {result.stderr.strip() or 'no error details'}")
